# Synthetic data generation for tool-calling fine-tuningThis notebook walks through fine-tuning a small model (`gpt-4.1-nano` / `gpt-4.1-mini`) to call a catalog of tools correctly, using training data **synthesized by the Foundry Data Generation API** from a static tool spec. No production agent traffic required.**You'll end up with**: a fine-tuned model that beats its baseline on tool-call correctness, and a leaderboard JSON file summarizing the experiment.**Time**: ~25–45 minutes (most of it waiting for FT jobs to train).**Cost**: ~$2–5 per run on the included Zava tool catalog (3 small candidates × a few thousand training tokens).---## What you need1. An Azure AI Foundry project. Set these env vars (the cell below will read them):   - `AZURE_AI_PROJECT_ENDPOINT` (e.g. `https://<resource>.services.ai.azure.com/api/projects/<project>`)   - `OPENAI_BASE_URL` (e.g. `https://<resource>.openai.azure.com/openai/v1`)   - `AZURE_OPENAI_API_KEY`2. A teacher model deployed on the project (e.g. `gpt-4.1`, `gpt-5.4`) for the datagen service to call.3. The `microsoft-foundry/fine-tuning` skill checked out locally. Set `FINETUNING_SKILL_PATH` to its `Skills/` folder.

In [ ]:
import os, json, subprocess, sysfrom pathlib import PathENDPOINT  = os.environ["AZURE_AI_PROJECT_ENDPOINT"]BASE_URL  = os.environ["OPENAI_BASE_URL"]API_KEY   = os.environ["AZURE_OPENAI_API_KEY"]SKILL     = Path(os.environ.get("FINETUNING_SKILL_PATH", "../../../Skills"))WORK      = Path("./run").resolve()WORK.mkdir(exist_ok=True)print(f"Project:  {ENDPOINT}")print(f"Base URL: {BASE_URL}")print(f"Skill:    {SKILL}")print(f"Work dir: {WORK}")

## 1. Convert OpenAI tools → OpenAPI 3.0The Foundry datagen service's `ToolUseFineTuning` recipe consumes an **OpenAPI 3.0 spec** as the tool catalog, not the OpenAI chat-completions tools array. The skill provides a converter; we use it on the bundled Zava 6-tool catalog.

In [ ]:
OPENAI_TOOLS = Path("fixtures/zava_tools_openai.json").resolve()OPENAPI_OUT  = WORK / "zava_tools_openapi.json"subprocess.run([    sys.executable, str(SKILL / "scripts" / "generate_dataset.py"),    "--tools-from", str(OPENAI_TOOLS),    "--tools-to-openapi-out", str(OPENAPI_OUT),], check=True)# Inspect the converted specspec = json.loads(OPENAPI_OUT.read_text())print(f"OpenAPI 3.0 spec: {len(spec['paths'])} operations")for path, op in spec["paths"].items():    method, rest = next(iter(op.items()))    print(f"  {method.upper():>4} {path}  ({rest.get('operationId', '?')})")

## 2. Upload the spec to your Foundry projectThe datagen API reads its source from a file in your project (purpose=`user_data`). We upload via the standard OpenAI files API.

In [ ]:
from openai import OpenAIclient = OpenAI(base_url=BASE_URL, api_key=API_KEY)with open(OPENAPI_OUT, "rb") as fh:    spec_file = client.files.create(file=("zava_tools_openapi.json", fh), purpose="user_data")print(f"Uploaded: {spec_file.id} ({spec_file.bytes:,} bytes)")

## 3. Generate synthetic training dataThe `ToolUseFineTuning` recipe writes ~N realistic user prompts and the correct tool invocations for each. The teacher model produces them; the service handles the prompting and validation.

In [ ]:
TEACHER_MODEL = "gpt-4.1"  # or gpt-5.4 if you have it deployedTASK_NAME     = "zava-tools"NUM_EXAMPLES  = 50# This shells out to the skill's `auto_finetune.py auto` with --datagen-backend foundry-file# and --datagen-recipe tool-use. The autopilot will:#   Phase 2: GENERATE     → call Foundry Data Generation API#   Phase 3: PREPARE      → split into train/val/test#   Phase 4: BASELINE     → score the un-tuned base model#   Phase 5: CANDIDATES   → design 3 candidate FT runs#   Phase 6: EXECUTE      → submit + monitor FT jobs#   Phase 7: EVALUATE     → score each FT'd model with tool-call comparison#   Phase 8: REVIEW       → SHIP if any candidate beats baseline by the lift threshold (default +5%, set in task_spec.json)cmd = [    sys.executable, str(SKILL / "scripts" / "auto_finetune.py"), "auto",    "--description", "Tool-calling assistant for the Zava retail Post-Purchase Resolution Desk. "                     "Helps customers with returns, exchanges, replacements, cancellations, and shipping disputes. "                     "Calls tools in order: get_order_details -> get_fulfillment_status -> check_resolution_policy "                     "-> [check_inventory if exchange] -> calculate_resolution -> submit_resolution.",    "--task-name", TASK_NAME,    "--model", "gpt-4.1-nano",    "--teacher", TEACHER_MODEL,    "--datagen-backend", "foundry-file",    "--datagen-file-id", spec_file.id,    "--datagen-recipe", "tool-use",    "--num-examples", str(NUM_EXAMPLES),    "--max-iterations", "1",    "--max-budget", "10",    "--work-dir", str(WORK),    "--tier", "globalStandard",]# Stream output so you can see phase transitions liveproc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)for line in proc.stdout:    print(line, end="")proc.wait()print(f"\n[exit {proc.returncode}]")

## 4. Inspect the leaderboardThe autopilot writes a `review_iter1.json` summarising the decision (`SHIP` / `ITERATE` / `STOP`) and the per-candidate scores. The winning model id can be invoked just like any deployed Azure OpenAI model.

In [ ]:
review = json.loads((WORK / "review_iter1.json").read_text())print(f"DECISION: {review['decision']}")print(f"REASON  : {review.get('reason', '(none)')}")print()print("Per-candidate leaderboard:")print(f"  {'candidate':<25} {'combined':>9} {'pass_rate':>10} {'lift_vs_base':>13}")print(f"  {'-'*25} {'-'*9} {'-'*10} {'-'*13}")for c in review.get("candidates", []):    lift = c.get("lift_pct")    lift_s = f"{lift:+.1f}%" if lift is not None else "—"    print(f"  {c['candidate']:<25} {c.get('combined', 0):>9.2f} {c.get('pass_rate', 0):>9.1f}% {lift_s:>13}")print()if review["decision"] == "SHIP":    print(f"✅ Shipped model: {review.get('winner', {}).get('model_id', '?')}")else:    print("ℹ️  No candidate met the lift threshold this iteration. See 'next_steps' in review.json for diagnostic recommendations.")

## 5. Try the fine-tuned modelIf a candidate shipped, you can call it the same way you'd call any Azure OpenAI deployment — with the `tools=` argument to test tool selection.

In [ ]:
if review["decision"] != "SHIP":    print("Skipping inference test — no winner this run.")else:    winner_model = review["winner"]["model_id"]    tools = json.loads(OPENAI_TOOLS.read_text())    test_prompt = ("My order #ZA-2057 from 3 weeks ago — the shoes arrived but the wrong size. "                   "Can I exchange for the next size up?")    resp = client.chat.completions.create(        model=winner_model,        messages=[{"role": "user", "content": test_prompt}],        tools=tools,        temperature=0,    )    msg = resp.choices[0].message    print(f"Tool calls emitted by {winner_model}:")    for tc in (msg.tool_calls or []):        print(f"  {tc.function.name}({tc.function.arguments})")    if not msg.tool_calls:        print(f"  (no tool calls; text response: {msg.content[:200]}...)")

## Cleanup (optional)If you don't want to keep the FT'd model around:```pythonclient.fine_tuning.jobs.cancel(<job_id>)  # if still running# Deployments created during eval are auto-deleted by the autopilot```The uploaded OpenAPI spec file counts against your project's 50-file quota; delete it when you're done:```pythonclient.files.delete(spec_file.id)```## Next steps- **Bring your own tools**: replace `fixtures/zava_tools_openai.json` with any OpenAI chat-completions tool array. The notebook handles the rest.- **More examples**: bump `NUM_EXAMPLES` to 200+ for higher-quality fine-tuning (longer datagen time, slightly higher cost).- **Higher lift target**: the autopilot's default lift threshold is +5%. For a stricter ship gate, edit `stopping_criteria.min_lift_pct` in the auto-generated task_spec.json (default 5.0).